## Computer Vision on Public Lands Webcams

#### Date: 11/12/2025
#### Author: Nineveh O'Connell

Goal: The goal of this notebook is to apply the YOLO computer vision tools to footage from the Acadia Sand Beach entrance station on Tuesday, August 26 to determine processing time at the entrance station.

In [1]:
#import libraries
import re
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import math

import time
from pathlib import Path
import os

import cv2
import yt_dlp
from ultralytics import YOLO
from collections import defaultdict
import supervision as sv
from bs4 import BeautifulSoup
import requests
from IPython.display import display, Image
from PIL import Image as Img
from PIL import ImageTk
from urllib.parse import urljoin


In [3]:
cap = cv2.VideoCapture(r"C:\Users\Nineveh.OConnell\OneDrive - DOT OST\volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection\ACAD Data Collection\4- Data Collection\Video Data\8.26.25_Park_Loop_Drive_GoProMax_09714\GS010034.360")

In [ ]:

# ----------------- User settings -----------------
VIDEO_PATH = r"C:\Users\Nineveh.OConnell\OneDrive - DOT OST\volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection\ACAD Data Collection\4- Data Collection\Video Data\8.26.25_Park_Loop_Drive_GoProMax_09714\GS010034.360"
OUT_RES = (1280, 720)   # perspective output resolution (w,h) — can reduce if too big
FOV = 60.0              # degrees (your chosen)
YAW = 0.0               # degrees
PITCH = -20.0           # degrees
ROLL = 180.0
# -------------------------------------------------

def rot_matrix(yaw_deg=0.0, pitch_deg=0.0, roll_deg=180.0):
    y = math.radians(yaw_deg); p = math.radians(pitch_deg); r = math.radians(roll_deg)
    cy = math.cos(y); sy = math.sin(y)
    cp = math.cos(p); sp = math.sin(p)
    cr = math.cos(r); sr = math.sin(r)
    Rz = np.array([[cr, -sr, 0.0],[sr, cr, 0.0],[0.0, 0.0, 1.0]])
    Rx = np.array([[1.0, 0.0, 0.0],[0.0, cp, -sp],[0.0, sp, cp]])
    Ry = np.array([[cy, 0.0, sy],[0.0, 1.0, 0.0],[-sy, 0.0, cy]])
    return Ry @ Rx @ Rz

def compute_remap(equi_w, equi_h, out_w, out_h, fov_deg, yaw_deg=0.0, pitch_deg=0.0, roll_deg=180.0):
    # Vectorized remap generator -> returns map_x,map_y float32 for cv2.remap
    f = 0.5 * out_w / math.tan(math.radians(fov_deg) / 2.0)
    u = np.linspace(0, out_w - 1, out_w)
    v = np.linspace(0, out_h - 1, out_h)
    uu, vv = np.meshgrid(u, v)
    x = (uu - (out_w - 1) / 2.0)
    y = - (vv - (out_h - 1) / 2.0)   # invert v because image y goes down
    z = np.full_like(x, f)
    vec = np.stack((x, y, z), axis=-1)
    vec = vec / (np.linalg.norm(vec, axis=-1, keepdims=True) + 1e-9)
    R = rot_matrix(yaw_deg, pitch_deg, roll_deg)
    flat = vec.reshape(-1, 3).T
    rotated = R @ flat
    rx = rotated[0, :].reshape(out_h, out_w)
    ry = rotated[1, :].reshape(out_h, out_w)
    rz = rotated[2, :].reshape(out_h, out_w)
    lon = np.arctan2(rx, rz)                # -pi..pi
    lat = np.arcsin(np.clip(ry, -1.0, 1.0)) # -pi/2..pi/2
    map_x = (lon + math.pi) / (2.0 * math.pi) * equi_w
    map_y = (math.pi / 2.0 - lat) / math.pi * equi_h
    return map_x.astype(np.float32), map_y.astype(np.float32)

def main():
    if not os.path.exists(VIDEO_PATH):
        raise FileNotFoundError("Update VIDEO_PATH to point to your video (or remux .360 to .mp4)")
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise RuntimeError("OpenCV cannot open the file. Try remuxing with ffmpeg to an mp4 and retry.")
    # read first frame to get equirectangular dims
    ok, first = cap.read()
    if not ok:
        raise RuntimeError("Cannot read frame from video.")
    H, W = first.shape[:2]
    out_w, out_h = OUT_RES
    # precompute remap maps for your chosen FOV/pitch/yaw
    map_x, map_y = compute_remap(W, H, out_w, out_h, FOV, YAW, PITCH, ROLL)
    model = YOLO("yolov8n.pt")  # change model path if needed

    # if you want to process every frame from start, reset
    # I'm skipping through a bunch to get to the point where cars actually are
    # could use some more toggling to not lose quality incredibly so, 
    # and incorporate a calculation of the timestamp from the frame count
    cap.set(cv2.CAP_PROP_POS_FRAMES, 13000)
    frame_idx = 0
    while True:
        ok, equi = cap.read()
        
        if frame_idx % 10 == 0:
            
            persp = cv2.remap(equi, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)

            # run ultralytics YOLO
            results = model(persp)  # returns a list-like object; each element corresponds to an image (here 1)
            r = results[0]
            # boxes in xyxy format (x1,y1,x2,y2) in perspective image coords
            if hasattr(r, "boxes") and len(r.boxes) > 0:
                xyxy = r.boxes.xyxy.cpu().numpy()       # shape (N,4)
                confs = r.boxes.conf.cpu().numpy()      # shape (N,)
                classes = r.boxes.cls.cpu().numpy().astype(int)  # shape (N,)
                for (x1,y1,x2,y2), conf, cls in zip(xyxy, confs, classes):
                    # compute center pixel of box in perspective image
                    cx = float((x1 + x2) / 2.0)
                    cy = float((y1 + y2) / 2.0)
                    # map center back to equirectangular using the remap arrays:
                    # note: map_x/y map from perspective pixel -> equirectangular coords
                    # need to round/clip indices since map arrays index by integer row/col
                    ix = int(np.clip(round(cy), 0, map_x.shape[0]-1))
                    iy = int(np.clip(round(cx), 0, map_x.shape[1]-1))
                    eq_x = float(map_x[ix, iy])
                    eq_y = float(map_y[ix, iy])
                    # eq_x, eq_y are pixel coords in the original equirectangular image
                    print(f"frame {frame_idx}: cls={cls} conf={conf:.2f} persp_center=({cx:.1f},{cy:.1f}) -> equi=({eq_x:.1f},{eq_y:.1f})")
                    # Optionally draw on equi (example; uncomment to visualize)
                    # cv2.circle(equi, (int(eq_x), int(eq_y)), 6, (0,0,255), -1)

            # (optional) show perspective for visual debug
            cv2.imshow("persp", persp)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


0: 384x640 (no detections), 208.8ms
Speed: 6.0ms preprocess, 208.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 188.6ms
Speed: 4.2ms preprocess, 188.6ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 211.8ms
Speed: 4.4ms preprocess, 211.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 174.4ms
Speed: 3.3ms preprocess, 174.4ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 176.3ms
Speed: 5.2ms preprocess, 176.3ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 170.3ms
Speed: 4.6ms preprocess, 170.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 141.2ms
Speed: 3.3ms preprocess, 141.2ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 158.4ms
Speed: 4.6ms prepr

error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\imgwarp.cpp:694: error: (-215:Assertion failed) !ssize.empty() in function 'cv::remapBilinear'


In [ ]:
# I want to document this somewhere, this chunk of code was really helpful
# for determining what parameters I want to look at the correct angle
# in a 360 video by creating a view where I can actively toggle
# things and see what the equivalent parameters are

"""
360 FOV tester
- Shows perspective crops for several FOVs side-by-side for quick visual testing.
- Vectorized remap computation (cached per parameter set).
- Keyboard controls:
    LEFT/RIGHT : prev/next frame (also drag slider by using frame_skip)
    UP/DOWN    : change yaw (in yaw_step increments)
    p/P        : change pitch +/- 5 degrees
    r          : toggle resolution list (cycles through provided out_resolutions)
    o          : toggle showing wrap indicator (not used here)
    s          : save current comparison image (filename printed)
    q / ESC    : quit

    # name window for viewer
    window_name = "Fullscreen Video"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
"""

import cv2
import numpy as np
import math
import time
import os

import tkinter as tk

def get_screen_size():
    # Try tkinter (cross-platform)
    try:
        root = tk.Tk()
        root.withdraw()
        w = root.winfo_screenwidth()
        h = root.winfo_screenheight()
        root.destroy()
        return w, h
    except Exception:
        # Fallback for Windows (ctypes)
        try:
            import ctypes
            user32 = ctypes.windll.user32
            return user32.GetSystemMetrics(0), user32.GetSystemMetrics(1)
        except Exception:
            # last resort: reasonable default
            return 1366, 768

# -------- User settings --------
VIDEO_PATH = r"C:\Users\Nineveh.OConnell\OneDrive - DOT OST\volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection\ACAD Data Collection\4- Data Collection\Video Data\8.26.25_Park_Loop_Drive_GoProMax_09714\GS010034.360"
# FOVs to test (degrees)
FOVS = [45.0, 60.0, 75.0] #60 is out winner
# Output resolutions to test (width, height)
OUT_RESOLUTIONS = [(640, 360), (960, 540), (1280, 720)]
# yaw step (degrees) for keyboard up/down increments
YAW_STEP = 30.0
# initial parameters
INIT_YAW = 0.0 #0 is out winner
INIT_PITCH = -20.0 #-20 is our winner
INIT_RES_INDEX = 2  # use index in OUT_RESOLUTIONS
FRAME_SKIP = 1  # how many frames to skip on next/prev
# --------------------------------

# ---- math helpers (vectorized) ----
def rot_matrix(yaw_deg=0.0, pitch_deg=0.0, roll_deg=0.0):
    yaw = math.radians(yaw_deg)
    pitch = math.radians(pitch_deg)
    roll = math.radians(roll_deg)
    cy = math.cos(yaw); sy = math.sin(yaw)
    cp = math.cos(pitch); sp = math.sin(pitch)
    cr = math.cos(roll); sr = math.sin(roll)
    Rz = np.array([[cr, -sr, 0.0],
                   [sr,  cr, 0.0],
                   [0.0, 0.0, 1.0]])
    Rx = np.array([[1.0, 0.0, 0.0],
                   [0.0, cp, -sp],
                   [0.0, sp,  cp]])
    Ry = np.array([[ cy, 0.0, sy],
                   [0.0, 1.0, 0.0],
                   [-sy, 0.0, cy]])
    R = Ry @ Rx @ Rz
    return R

def compute_remap(equi_w, equi_h, out_w, out_h, fov_deg, yaw_deg=0.0, pitch_deg=0.0, roll_deg=0.0):
    """
    Vectorized computation of map_x, map_y (float32) for cv2.remap
    map_x, map_y shape = (out_h, out_w)
    out image sample: remapped = cv2.remap(equi, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)
    """
    # focal length in pixels (based on width)
    f = 0.5 * out_w / math.tan(math.radians(fov_deg) / 2.0)
    # pixel grid in image plane
    u = np.linspace(0, out_w - 1, out_w)
    v = np.linspace(0, out_h - 1, out_h)
    uu, vv = np.meshgrid(u, v)  # shape (out_h, out_w)
    # camera coords: center at (out_w/2, out_h/2), y down needs inversion
    x = (uu - (out_w - 1) / 2.0)
    y = - (vv - (out_h - 1) / 2.0)
    z = np.full_like(x, f)
    # normalize direction vectors
    vec = np.stack((x, y, z), axis=-1)  # (out_h, out_w, 3)
    norm = np.linalg.norm(vec, axis=-1, keepdims=True)
    vec = vec / (norm + 1e-9)
    # rotate to world using rotation matrix
    R = rot_matrix(yaw_deg, pitch_deg, roll_deg)  # (3,3)
    # apply rotation: for each vector v' = R @ v
    # reshape to (n,3), dot, reshape back
    flat = vec.reshape(-1, 3).T  # (3, n)
    rotated = R @ flat           # (3, n)
    rx = rotated[0, :].reshape(out_h, out_w)
    ry = rotated[1, :].reshape(out_h, out_w)
    rz = rotated[2, :].reshape(out_h, out_w)
    # convert direction to lon/lat:
    # lon = atan2(x, z), lat = asin(y)  (vec normalized)
    lon = np.arctan2(rx, rz)  # (-pi, pi)
    lat = np.arcsin(np.clip(ry, -1.0, 1.0))  # (-pi/2, pi/2)
    # map to equirectangular pixels
    map_x = (lon + math.pi) / (2.0 * math.pi) * equi_w
    map_y = (math.pi / 2.0 - lat) / math.pi * equi_h
    # Ensure float32 for cv2.remap
    return map_x.astype(np.float32), map_y.astype(np.float32)

# cache remaps for reuse across frames
remap_cache = {}

def get_remap_for_params(equi_w, equi_h, out_w, out_h, fov_deg, yaw_deg, pitch_deg, roll_deg=0.0):
    key = (equi_w, equi_h, out_w, out_h, float(fov_deg), float(yaw_deg), float(pitch_deg), float(roll_deg))
    if key not in remap_cache:
        map_x, map_y = compute_remap(equi_w, equi_h, out_w, out_h, fov_deg, yaw_deg, pitch_deg, roll_deg)
        remap_cache[key] = (map_x, map_y)
    return remap_cache[key]

# ---- main UI / loop ----
def build_comparison_frame(equi_frame, yaw_deg, pitch_deg, res_index):
    H, W = equi_frame.shape[:2]
    out_w, out_h = OUT_RESOLUTIONS[res_index]
    crops = []
    for fov in FOVS:
        map_x, map_y = get_remap_for_params(W, H, out_w, out_h, fov, yaw_deg, pitch_deg)
        out = cv2.remap(equi_frame, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)
        # annotate FOV text
        label = f"{int(fov)}°  {out_w}x{out_h}"
        cv2.putText(out, label, (8, out_h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2, cv2.LINE_AA)
        crops.append(out)
    # arrange crops horizontally
    spacer = 6
    total_w = sum(c.shape[1] for c in crops) + spacer * (len(crops) - 1)
    total_h = max(c.shape[0] for c in crops)
    canvas = np.zeros((total_h, total_w, 3), dtype=np.uint8)
    x = 0
    for c in crops:
        h, w = c.shape[:2]
        canvas[0:h, x:x+w] = c
        x += w + spacer
    # add top info
    info = f"Yaw={yaw_deg:.1f} Pitch={pitch_deg:.1f}  (res idx {res_index} => {out_w}x{out_h})  FOVs={FOVS}"
    cv2.putText(canvas, info, (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 2, cv2.LINE_AA)
    return canvas

def main():
    global remap_cache
    video_path = VIDEO_PATH
    if not os.path.exists(video_path):
        print("Video path does not exist. Update VIDEO_PATH at top of script.")
        return

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("OpenCV cannot open the file directly. Try remuxing to .mp4 with ffmpeg, then re-run.")
        print('ffmpeg -i "GS010034.360" -c copy "GS010034_remux.mp4"')
        return

    # read some info
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if cap.get(cv2.CAP_PROP_FRAME_COUNT) > 0 else None
    print("Opened video:", video_path, "frames:", frame_count)
    frame_idx = 0
    yaw = INIT_YAW
    pitch = INIT_PITCH
    res_index = INIT_RES_INDEX
    saved_count = 0

    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            # clamp
            if frame_count:
                frame_idx = max(0, min(frame_count - 1, frame_idx))
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                ok, frame = cap.read()
            if not ok:
                print("Reached end or cannot read frame.")
                break

        # build comparison display
        comp = build_comparison_frame(frame, yaw, pitch, res_index)
        # --- scale to screen if needed ---
        screen_w, screen_h = get_screen_size()
        # leave some margin for taskbar/OS chrome
        margin_w, margin_h = 80, 120
        max_w = screen_w - margin_w
        max_h = screen_h - margin_h

        ch, cw = comp.shape[:2]
        scale = min(1.0, max_w / cw, max_h / ch)  # <= 1.0
        if scale < 1.0:
            disp_w = max(1, int(cw * scale))
            disp_h = max(1, int(ch * scale))
            disp = cv2.resize(comp, (disp_w, disp_h), interpolation=cv2.INTER_AREA)
        else:
            disp = comp.copy()

        window_name = "360 FOV Tester (press q to quit)"
        cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
        # ensure window is the right size (some backends ignore resize unless set)
        cv2.resizeWindow(window_name, disp.shape[1], disp.shape[0])
        cv2.imshow(window_name, disp)
        key = cv2.waitKey(0) & 0xFF

        if key == ord('q') or key == 27:
            break
        elif key == ord('s'):
            # save current canvas
            fname = f"fov_test_frame{frame_idx}_yaw{int(yaw)}_pitch{int(pitch)}_res{res_index}.png"
            cv2.imwrite(fname, comp)
            saved_count += 1
            print("Saved:", fname)
        elif key == ord('j'):  # LEFT arrow
            frame_idx = max(0, frame_idx - max(1, FRAME_SKIP))
        elif key == ord('l'):  # RIGHT arrow
            print("i see right arrow")
            frame_idx = frame_idx + max(1, FRAME_SKIP)
            if frame_count:
                frame_idx = min(frame_idx, frame_count - 1)
        elif key == ord('i'):  # UP arrow -> increment yaw positively. Ok Yaw of 0 seems right
            print("i see up arrow")
            yaw = (yaw + YAW_STEP) % 360.0
        elif key == ord('k'):  # DOWN arrow -> decrement yaw
            yaw = (yaw - YAW_STEP) % 360.0
        elif key == ord('p'):
            pitch = np.clip(pitch + 5.0, -89.0, 89.0)
        elif key == ord('o'): # pitch of -20 looks pretty good, and fov of 60
            pitch = np.clip(pitch - 5.0, -89.0, 89.0)
        elif key == ord('r'):
            res_index = (res_index + 1) % len(OUT_RESOLUTIONS)
        else:
            # unrecognized key: pass
            pass

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Opened video: C:\Users\Nineveh.OConnell\OneDrive - DOT OST\volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection\ACAD Data Collection\4- Data Collection\Video Data\8.26.25_Park_Loop_Drive_GoProMax_09714\GS010034.360 frames: 14460


## YOLO modeling of video

In [2]:
def list_files_pathlib(directory_path_str):
    """
    Lists all files in the specified directory using pathlib.
    """
    directory_path = Path(directory_path_str)
    files = [str(p.resolve()) for p in directory_path.iterdir() if p.is_file()]
    return files

# Example usage (for current directory):
p_dir_base = "C:/Users/Nineveh.OConnell/OneDrive - DOT OST/volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection/ACAD Data Collection/4- Data Collection/Video Data/8.28.25 Eagle Lake Dashcam/VIDEO_F/"
all_files_pathlib = list_files_pathlib(p_dir_base)



In [6]:

if not cap.isOpened():
    print("Error opening video file")

# Get dimensions
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Get length
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if fps > 0:
    duration_seconds = frame_count / fps
else:
    duration_seconds = 0

print(width)
print(height)
print(duration_seconds)

4096
1344
482.482


In [2]:
# Load the YOLO model
model = YOLO('yolo11l.pt')

class_list = model.names 

In [ ]:
video_base_name = os.path.basename(video_path)


In [4]:

def get_video_properties_cv2(filename):
    video = cv2.VideoCapture(filename)
    
    if not video.isOpened():
        print(f"Error opening video file: {filename}")
        return None

    # Get dimensions
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Get length
    fps = video.get(cv2.CAP_PROP_FPS)
    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if fps > 0:
        duration_seconds = frame_count / fps
    else:
        duration_seconds = 0

    video.release()

    return {
        "width": width,
        "height": height,
        "duration_seconds": duration_seconds
    }

# Example Usage
properties = get_video_properties_cv2(all_files_pathlib[0])
if properties:
    print(f"Dimensions: {properties['width']}x{properties['height']}")
    print(f"Length (seconds): {properties['duration_seconds']:.2f}s")


Dimensions: 2560x1440
Length (seconds): 60.04s


In [8]:
# 6) extract numeric confidence from a string like "label (0.82)" into confidence_numeric
#    regex captures the number inside parentheses (first occurrence)
# this will define that function
def extract_confidence(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r"\(([^)]+)\)", str(s))
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return np.nan
    return np.nan

In [ ]:
# ---- helper math functions ----
def equirectangular_pixel_to_lonlat(x, y, W, H):
    # x in [0,W), y in [0,H)
    lon = (x / W) * 2.0 * math.pi - math.pi   # [-pi, pi]
    lat = math.pi/2.0 - (y / H) * math.pi     # [-pi/2, pi/2]
    return lon, lat

def lonlat_to_equirectangular_pixel(lon, lat, W, H):
    x = (lon + math.pi) / (2.0 * math.pi) * W
    y = (math.pi/2.0 - lat) / math.pi * H
    return x, y

def perspective_pixel_to_direction(u, v, w, h, fov_deg):
    # maps pixel (u,v) in perspective image to direction vector in camera coords
    # center at (w/2, h/2)
    f = 0.5 * w / math.tan(math.radians(fov_deg) / 2.0)
    x = (u - w/2.0)
    y = (v - h/2.0)
    z = f
    v3 = np.array([x, -y, z], dtype=np.float64)  # -y because image y down
    v3 = v3 / np.linalg.norm(v3)
    return v3

def rotate_vector(v, yaw, pitch, roll):
    # yaw (around Y), pitch (around X), roll (around Z); angles in radians
    cy = math.cos(yaw); sy = math.sin(yaw)
    cp = math.cos(pitch); sp = math.sin(pitch)
    cr = math.cos(roll); sr = math.sin(roll)

    # rotation matrices (apply roll, pitch, yaw)
    Rz = np.array([[cr,-sr,0],[sr,cr,0],[0,0,1]])
    Rx = np.array([[1,0,0],[0,cp,-sp],[0,sp,cp]])
    Ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
    R = Ry @ Rx @ Rz
    return R @ v

def direction_to_lonlat(v):
    x,y,z = v
    lon = math.atan2(x, z)        # note: depends on coordinate conventions
    r = math.sqrt(x*x + z*z)
    lat = math.atan2(y, r)
    return lon, lat

# ---- function to generate perspective view from equirectangular frame ----
def equirectangular_to_perspective(equi, yaw_deg=0.0, pitch_deg=-20.0, roll_deg=0.0, fov_deg=60.0, w=1280, h=720):
    H, W = equi.shape[:2]
    yaw = math.radians(yaw_deg); pitch = math.radians(pitch_deg); roll = math.radians(roll_deg)
    # Build maps
    map_x = np.zeros((h, w), dtype=np.float32)
    map_y = np.zeros((h, w), dtype=np.float32)

    for v in range(h):
        for u in range(w):
            dir_cam = perspective_pixel_to_direction(u, v, w, h, fov_deg)
            dir_world = rotate_vector(dir_cam, yaw, pitch, roll)
            lon, lat = direction_to_lonlat(dir_world)
            px, py = lonlat_to_equirectangular_pixel(lon, lat, W, H)
            # clamp
            map_x[v, u] = px
            map_y[v, u] = py

    perspective = cv2.remap(equi, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)
    return perspective, map_x, map_y


W_out, H_out = 1280, 720
fov = 60.0

cap = cv2.VideoCapture(r"C:\Users\Nineveh.OConnell\OneDrive - DOT OST\volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection\ACAD Data Collection\4- Data Collection\Video Data\8.26.25_Park_Loop_Drive_GoProMax_09714\GS010034.360")
cap.set(cv2.CAP_PROP_POS_FRAMES, 3500)

window_name = "Fullscreen Video"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

while True:
    ok, frame = cap.read()


    frame_num = int(cap.get(cv2.CAP_PROP_POS_FRAMES))


    # If detections occur, do operations only beyond frame 3000 
    if (frame_num >= 3500) and (frame_num % 10 == 0):

        print(frame_num)
        
        if not ok:
            print(f"yo she just failed at {frame_num}")
            break

        # create a perspective view at desired yaw/pitch
        yaw_deg = 0.0    # try 0, 45, 90, ... to cover different directions
        persp, map_x, map_y = equirectangular_to_perspective(frame, yaw_deg=yaw_deg, pitch_deg=-20.0, fov_deg=fov, w=W_out, h=H_out)

        # run YOLO on the perspective frame
        results = model.track(persp, classes = [0,1,2,3,5,7,11], persist = True)  # ultralytics returns results list
        # iterate detections (this API depends on ultralytics version)
        for r in results:
            boxes = r.boxes.cpu().numpy() if hasattr(r, "boxes") else []
            # Here: boxes array typically [x1,y1,x2,y2,score,class]
            for box in boxes:
                x1,y1,x2,y2 = map(int, box[:4])
                cx = (x1 + x2) / 2.0
                cy = (y1 + y2) / 2.0
                # map center back to equirectangular pixel using the remap maps
                # map_x and map_y are float arrays mapping target->source
                eq_x = float(map_x[int(cy), int(cx)])
                eq_y = float(map_y[int(cy), int(cx)])
                # eq_x, eq_y are pixel coords in the original equirectangular frame
                print("Detected class:", box[5] if len(box) > 5 else None, "center in equirectangular px:", (eq_x, eq_y))
        # optional: show or save results; break loop for demo
        cv2.imshow(window_name, persp); 
    
    if cv2.waitKey(1) & 0xFF == ord('q'): 
        break
cap.release()

cv2.destroyAllWindows()

3510

0: 384x640 (no detections), 352.4ms
Speed: 2.0ms preprocess, 352.4ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)
3520

0: 384x640 (no detections), 423.5ms
Speed: 1.9ms preprocess, 423.5ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)


In [ ]:
video_path = r"C:\Users\Nineveh.OConnell\OneDrive - DOT OST\volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection\ACAD Data Collection\4- Data Collection\Video Data\8.26.25_Park_Loop_Drive_GoProMax_09714\GS010034.360"

# Open video
cap = cv2.VideoCapture(video_path)
frame_rate = cap.get(cv2.CAP_PROP_FPS)
n_frames_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
curr_frame_num = 0

# name window for viewer
window_name = "Fullscreen Video"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

# to save results
resultsList = []

# the approach of using a while looks and checking for success isn't working
# instead let's use a different while condition
# this was the condition before: while cap.isOpened()
#print(f"Video {video_base_name} is running through computer vision model...\n")
while curr_frame_num < n_frames_total:
    ret, frame = cap.read()

    # Get current frame number
    frame_num = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
    curr_frame_num = frame_num
    
    # If detections occur, do operations for every tenth frame 
    if frame_num % 10 == 0:

        if not ret:
            print("this is the for-loop internal call of")
            print("Video completed or error reading frame.")
            break

        # Process frame for detections
        results = model.track(frame, classes = [0,1,2,3,5,7,11], persist = True)

        if len(results) > 0:

            timestamp = frame_num / frame_rate
            cv2.putText(frame, f'Timestamp: {timestamp:.2f}s', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 
                        1, (255, 255, 255), 2, cv2.LINE_AA)
        
            # Here you can save the frame or timestamp if needed
            for r in results:

                if r.boxes.id is not None:
                    boxes = r.boxes.xyxy.cpu()  # Boxes object for bbox outputs
                    print(boxes)
                    track_ids = r.boxes.id.int().cpu().tolist()
                    class_indices = r.boxes.cls.int().cpu().tolist()
                    confidences = r.boxes.conf.cpu()

                    # Loop through each detected object
                    for box, track_id, class_idx, conf in zip(boxes, track_ids, class_indices, confidences):
                        x1, y1, x2, y2 = map(int, box)
                        cx = (x1 + x2) // 2  # Calculate the center point
                        cy = (y1 + y2) // 2            

                        class_name = class_list[class_idx]

                        cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                        
                        cv2.putText(frame, f"ID: {track_id} {class_name}", (x1, y1 - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) 

                        dfKeyFeatures = pd.DataFrame({'id' : [track_id], 
                                                    'class' : [class_name], 
                                                    'confidence' : [conf], 
                                                    'cx' : [cx], 
                                                    'cy' : [cy], 
                                                    'bottomLeftx' : [x1],
                                                    'bottomLefty' : [y1],
                                                    'upperRightx' : [x2],
                                                    'upperRighty' : [y2],
                                                    'timestamp' : [timestamp]})
                        resultsList.append(dfKeyFeatures)

            # Display the frame
            cv2.rectangle(frame, (1350, 500), (2200, 1000), (100, 80, 250), 2)

            cv2.imshow(window_name, frame)
        
    # if manual override, break and move to next file in path
    if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()




0: 224x640 2 persons, 1 truck, 422.5ms
Speed: 5.4ms preprocess, 422.5ms inference, 12.3ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 1 person, 6 cars, 1 truck, 242.9ms
Speed: 2.2ms preprocess, 242.9ms inference, 3.5ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 1 person, 1 car, 242.8ms
Speed: 2.6ms preprocess, 242.8ms inference, 1.7ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 1 person, 3 cars, 219.8ms
Speed: 2.2ms preprocess, 219.8ms inference, 0.8ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 1 person, 259.7ms
Speed: 2.5ms preprocess, 259.7ms inference, 2.4ms postprocess per image at shape (1, 3, 224, 640)
tensor([[3948.8418,  560.7299, 4094.4521, 1002.6678]])

0: 224x640 2 persons, 369.3ms
Speed: 2.6ms preprocess, 369.3ms inference, 1.8ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 1 person, 246.9ms
Speed: 2.4ms preprocess, 246.9ms inference, 1.3ms postprocess per image at shape (1, 3, 224, 640)
tenso

In [11]:
# compile results
combined_df = pd.concat(resultsList)

# 6) extract numeric confidence from a string like "label (0.82)" into confidence_numeric
#    regex captures the number inside parentheses (first occurrence)
def extract_confidence(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r"\(([^)]+)\)", str(s))
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return np.nan
    return np.nan

combined_df["confidence_numeric"] = combined_df["confidence"].apply(extract_confidence)

# export results to csv
#video_base_name = os.path.basename(video_path)
#combined_df.to_csv(f"C:/Users/Nineveh.OConnell/OneDrive - DOT OST/volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection/ACAD Data Collection/7- Video Analysis/ParkLoopRd/cv_output{video_base_name}.csv", index=False) 

ValueError: No objects to concatenate

## Defining Lanes

In [73]:
lane_boundaries = pd.DataFrame({
    "spot_id": np.arange(1, 5),
    "minx": [660, 1020, 1180, 1300],
    "maxx": [660, 920, 1150, 1300],
    "miny": [630, 560, 530, 450],
    "maxy" : [800, 650, 600, 400]
}).sort_values("minx").reset_index(drop=True)


Assign to parking spots, calling the parking spot -1 in the case of being in the roadway

In [74]:
combined_df['lane_id_minx'] = pd.cut(combined_df['cx'], bins = lane_boundaries['minx'], labels = np.arange(1,4), right = True)
combined_df['lane_id_maxx'] = pd.cut(combined_df['cx'], bins = lane_boundaries['maxx'], labels = np.arange(1,4), right = True)
#combined_df['lane_id_miny'] = pd.cut(combined_df['cy'], bins = lane_boundaries['miny'], labels = np.arange(1,4)[::-1], right = True)
#combined_df['lane_id_maxy'] = pd.cut(combined_df['cy'], bins = lane_boundaries['maxx'], labels = np.arange(1,4)[::-1], right = True)


Make confidence into a numeric variable and only keep instances with confidence over 0.35. From spot checking, instances with lower confidence are not really vehicles.

In [75]:
# 6) extract numeric confidence from a string like "label (0.82)" into confidence_numeric
#    regex captures the number inside parentheses (first occurrence)
def extract_confidence(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r"\(([^)]+)\)", str(s))
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return np.nan
    return np.nan

combined_df["confidence_numeric"] = combined_df["confidence"].apply(extract_confidence)

# # keep only rows where confidence is 0.35 or greater
# willow_creek_vehicles = combined_df[combined_df["confidence_numeric"] > 0.35]
# # Clean up column
# willow_creek_vehicles = willow_creek_vehicles.drop(columns=["confidence"])
# # make timestamp actual date time object, and turn id into a string
# willow_creek_vehicles['timestamp_dt'] = pd.to_datetime(willow_creek_vehicles['timestamp'], format='%Y-%m-%d %H-%M-%S')



In [ ]:
# export results to csv
combined_df.to_csv(f"C:/Users/Nineveh.OConnell/OneDrive - DOT OST/volpe-proj-VXAGA1-NPS NERO - ACAD Data Collection/ACAD Data Collection/7- Video Analysis/ParkLoopRd/cv_output{video_file_name}.csv", index=False) 